# 07 - Gráficos del log de seguridad web (`access.csv`)

Este notebook carga el dataset `access.csv`, prepara las fechas y genera una batería amplia de gráficos de análisis de seguridad.

**Resultado:** cada gráfico se muestra en el notebook y también se guarda automáticamente en el directorio `graficos`.

> Coloca este notebook en el mismo directorio donde está `access.csv` antes de ejecutarlo.

In [ ]:
# ============================================================
# 1. Importación de librerías
# ============================================================
# pandas: lectura, limpieza y agrupación de datos.
# matplotlib: generación de gráficos.
# pathlib: manejo de rutas de archivos y carpetas.
# textwrap: ayuda para cortar etiquetas largas en los gráficos.

from pathlib import Path
import textwrap

import pandas as pd
import matplotlib.pyplot as plt

# Para que los gráficos se vean dentro del notebook.
%matplotlib inline

# Configuración general de matplotlib.
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["font.size"] = 10

# Archivo de entrada y carpeta de salida.
DATASET = Path("access.csv")
DIR_GRAFICOS = Path("graficos")
DIR_GRAFICOS.mkdir(exist_ok=True)

print(f"Dataset esperado: {DATASET.resolve()}")
print(f"Carpeta de gráficos: {DIR_GRAFICOS.resolve()}")

In [ ]:
# ============================================================
# 2. Carga del dataset
# ============================================================
# Se carga el archivo CSV. Si el archivo no existe en el directorio actual,
# se mostrará un error claro para corregir la ubicación.

if not DATASET.exists():
    raise FileNotFoundError(
        f"No se encontró {DATASET}. Coloca el archivo access.csv en el mismo directorio del notebook."
    )

df = pd.read_csv(DATASET)

print("Dimensiones del dataset:", df.shape)
display(df.head())
print("Columnas encontradas:")
print(df.columns.tolist())

In [ ]:
# ============================================================
# 3. Preparación y enriquecimiento de datos
# ============================================================
# El campo fecha viene en formato de log Apache/Nginx:
# ejemplo: 06/May/2026:09:07:52 -0500
# Lo convertimos a datetime para poder graficar por tiempo.

datos = df.copy()

# Conversión robusta de fecha. utc=True ayuda a manejar el offset -0500.
datos["fecha_dt"] = pd.to_datetime(
    datos["fecha"],
    format="%d/%b/%Y:%H:%M:%S %z",
    errors="coerce",
    utc=True
)

# Convertimos la fecha a zona horaria local de Perú para el análisis visual.
datos["fecha_dt"] = datos["fecha_dt"].dt.tz_convert("America/Lima")

# Variables derivadas útiles para análisis temporal.
datos["fecha_dia"] = datos["fecha_dt"].dt.date
datos["hora"] = datos["fecha_dt"].dt.hour
datos["dia_semana"] = datos["fecha_dt"].dt.day_name()
datos["minuto"] = datos["fecha_dt"].dt.floor("min")
datos["es_ataque_label"] = datos["es_ataque"].map({0: "Normal", 1: "Ataque"}).fillna(datos["es_ataque"].astype(str))

print("Valores nulos por columna:")
display(datos.isna().sum().to_frame("nulos"))

display(datos.head())

In [ ]:
# ============================================================
# 4. Funciones auxiliares para mostrar y guardar gráficos
# ============================================================

def guardar_mostrar(nombre_archivo: str):
    # Guarda el gráfico actual en la carpeta graficos y luego lo muestra.
    ruta = DIR_GRAFICOS / nombre_archivo
    plt.tight_layout()
    plt.savefig(ruta, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Gráfico guardado en: {ruta}")


def envolver_etiquetas(etiquetas, ancho=18):
    # Corta etiquetas largas para que no se monten en el eje X.
    return ["\n".join(textwrap.wrap(str(e), width=ancho)) for e in etiquetas]


def grafico_barras_conteo(serie, titulo, xlabel, ylabel, archivo, top=None, rotacion=45):
    # Genera un gráfico de barras a partir de una serie de conteos.
    conteos = serie.value_counts(dropna=False)
    if top:
        conteos = conteos.head(top)
    
    plt.figure(figsize=(12, 6))
    conteos.plot(kind="bar")
    plt.title(titulo)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(range(len(conteos.index)), envolver_etiquetas(conteos.index), rotation=rotacion, ha="right")
    guardar_mostrar(archivo)


def grafico_heatmap(tabla, titulo, archivo):
    # Genera un heatmap simple con matplotlib desde una tabla dinámica.
    plt.figure(figsize=(12, 7))
    plt.imshow(tabla.values, aspect="auto")
    plt.title(titulo)
    plt.xlabel(tabla.columns.name if tabla.columns.name else "Columnas")
    plt.ylabel(tabla.index.name if tabla.index.name else "Filas")
    plt.xticks(range(len(tabla.columns)), envolver_etiquetas(tabla.columns, 12), rotation=45, ha="right")
    plt.yticks(range(len(tabla.index)), envolver_etiquetas(tabla.index, 12))
    plt.colorbar(label="Cantidad de eventos")
    guardar_mostrar(archivo)

## 5. Análisis general del dataset

In [ ]:
# 5.1 Eventos normales vs ataques

grafico_barras_conteo(
    datos["es_ataque_label"],
    titulo="Eventos normales vs eventos de ataque",
    xlabel="Clasificación",
    ylabel="Cantidad de eventos",
    archivo="01_eventos_normales_vs_ataques.png",
    rotacion=0
)

In [ ]:
# 5.2 Distribución por tipo de ataque

grafico_barras_conteo(
    datos["tipo_ataque"],
    titulo="Distribución por tipo de ataque",
    xlabel="Tipo de ataque",
    ylabel="Cantidad de eventos",
    archivo="02_distribucion_tipo_ataque.png"
)

In [ ]:
# 5.3 Distribución por método HTTP

grafico_barras_conteo(
    datos["metodo"],
    titulo="Distribución de métodos HTTP",
    xlabel="Método HTTP",
    ylabel="Cantidad de solicitudes",
    archivo="03_distribucion_metodos_http.png",
    rotacion=0
)

In [ ]:
# 5.4 Distribución por código de estado HTTP

grafico_barras_conteo(
    datos["codigo_estado"].astype(str),
    titulo="Distribución de códigos de estado HTTP",
    xlabel="Código de estado",
    ylabel="Cantidad de respuestas",
    archivo="04_distribucion_codigos_estado.png",
    rotacion=0
)

In [ ]:
# 5.5 Distribución de tamaños de respuesta

plt.figure(figsize=(12, 6))
plt.hist(datos["tamano_respuesta"].dropna(), bins=40)
plt.title("Distribución del tamaño de respuesta")
plt.xlabel("Tamaño de respuesta en bytes")
plt.ylabel("Cantidad de eventos")
guardar_mostrar("05_histograma_tamano_respuesta.png")

In [ ]:
# 5.6 Boxplot del tamaño de respuesta por normal/ataque

normal = datos.loc[datos["es_ataque_label"] == "Normal", "tamano_respuesta"].dropna()
ataque = datos.loc[datos["es_ataque_label"] == "Ataque", "tamano_respuesta"].dropna()

plt.figure(figsize=(10, 6))
plt.boxplot([normal, ataque], tick_labels=["Normal", "Ataque"])
plt.title("Comparación del tamaño de respuesta: normal vs ataque")
plt.xlabel("Clasificación")
plt.ylabel("Tamaño de respuesta en bytes")
guardar_mostrar("06_boxplot_tamano_respuesta_normal_vs_ataque.png")

## 6. Análisis temporal

In [ ]:
# 6.1 Eventos por día

eventos_dia = datos.groupby("fecha_dia").size()

plt.figure(figsize=(12, 6))
eventos_dia.plot(kind="line", marker="o")
plt.title("Cantidad de eventos por día")
plt.xlabel("Día")
plt.ylabel("Cantidad de eventos")
plt.xticks(rotation=45)
guardar_mostrar("07_eventos_por_dia.png")

In [ ]:
# 6.2 Ataques por día

ataques_dia = datos[datos["es_ataque"] == 1].groupby("fecha_dia").size()

plt.figure(figsize=(12, 6))
ataques_dia.plot(kind="line", marker="o")
plt.title("Cantidad de ataques por día")
plt.xlabel("Día")
plt.ylabel("Cantidad de ataques")
plt.xticks(rotation=45)
guardar_mostrar("08_ataques_por_dia.png")

In [ ]:
# 6.3 Eventos por hora del día

eventos_hora = datos.groupby("hora").size().reindex(range(24), fill_value=0)

plt.figure(figsize=(12, 6))
eventos_hora.plot(kind="bar")
plt.title("Cantidad de eventos por hora del día")
plt.xlabel("Hora")
plt.ylabel("Cantidad de eventos")
plt.xticks(rotation=0)
guardar_mostrar("09_eventos_por_hora.png")

In [ ]:
# 6.4 Ataques por hora del día

ataques_hora = datos[datos["es_ataque"] == 1].groupby("hora").size().reindex(range(24), fill_value=0)

plt.figure(figsize=(12, 6))
ataques_hora.plot(kind="bar")
plt.title("Cantidad de ataques por hora del día")
plt.xlabel("Hora")
plt.ylabel("Cantidad de ataques")
plt.xticks(rotation=0)
guardar_mostrar("10_ataques_por_hora.png")

In [ ]:
# 6.5 Comparación de eventos normales y ataques por hora

tabla_hora = pd.crosstab(datos["hora"], datos["es_ataque_label"]).reindex(range(24), fill_value=0)

plt.figure(figsize=(12, 6))
tabla_hora.plot(kind="bar", stacked=True, ax=plt.gca())
plt.title("Eventos normales y ataques por hora")
plt.xlabel("Hora")
plt.ylabel("Cantidad de eventos")
plt.xticks(rotation=0)
guardar_mostrar("11_normales_vs_ataques_por_hora.png")

In [ ]:
# 6.6 Heatmap de ataques por día y hora

tabla_dia_hora = pd.pivot_table(
    datos[datos["es_ataque"] == 1],
    index="fecha_dia",
    columns="hora",
    values="ip_origen",
    aggfunc="count",
    fill_value=0
)

if not tabla_dia_hora.empty:
    grafico_heatmap(tabla_dia_hora, "Heatmap de ataques por día y hora", "12_heatmap_ataques_dia_hora.png")
else:
    print("No hay ataques para graficar el heatmap día/hora.")

## 7. Análisis por IP, URL y User-Agent

In [ ]:
# 7.1 Top 20 IPs de origen con más eventos

grafico_barras_conteo(
    datos["ip_origen"],
    titulo="Top 20 IPs de origen con más eventos",
    xlabel="IP de origen",
    ylabel="Cantidad de eventos",
    archivo="13_top_20_ips_eventos.png",
    top=20
)

In [ ]:
# 7.2 Top 20 IPs de origen con más ataques

ataques = datos[datos["es_ataque"] == 1]

grafico_barras_conteo(
    ataques["ip_origen"],
    titulo="Top 20 IPs de origen con más ataques",
    xlabel="IP de origen",
    ylabel="Cantidad de ataques",
    archivo="14_top_20_ips_ataques.png",
    top=20
)

In [ ]:
# 7.3 Top 20 URLs más solicitadas

grafico_barras_conteo(
    datos["url"],
    titulo="Top 20 URLs más solicitadas",
    xlabel="URL",
    ylabel="Cantidad de solicitudes",
    archivo="15_top_20_urls_solicitadas.png",
    top=20
)

In [ ]:
# 7.4 Top 20 URLs más atacadas

grafico_barras_conteo(
    ataques["url"],
    titulo="Top 20 URLs más atacadas",
    xlabel="URL",
    ylabel="Cantidad de ataques",
    archivo="16_top_20_urls_atacadas.png",
    top=20
)

In [ ]:
# 7.5 Distribución por User-Agent

grafico_barras_conteo(
    datos["user_agent"],
    titulo="Distribución por User-Agent",
    xlabel="User-Agent",
    ylabel="Cantidad de eventos",
    archivo="17_distribucion_user_agent.png",
    top=20
)

In [ ]:
# 7.6 User-Agent más usados en ataques

grafico_barras_conteo(
    ataques["user_agent"],
    titulo="User-Agent más frecuentes en ataques",
    xlabel="User-Agent",
    ylabel="Cantidad de ataques",
    archivo="18_user_agent_ataques.png",
    top=20
)

## 8. Gráficos cruzados para análisis de seguridad

In [ ]:
# 8.1 Método HTTP vs tipo de ataque

tabla_metodo_ataque = pd.crosstab(datos["metodo"], datos["tipo_ataque"])
grafico_heatmap(tabla_metodo_ataque, "Método HTTP vs tipo de ataque", "19_heatmap_metodo_vs_tipo_ataque.png")

In [ ]:
# 8.2 Código de estado vs tipo de ataque

tabla_estado_ataque = pd.crosstab(datos["codigo_estado"].astype(str), datos["tipo_ataque"])
grafico_heatmap(tabla_estado_ataque, "Código de estado HTTP vs tipo de ataque", "20_heatmap_estado_vs_tipo_ataque.png")

In [ ]:
# 8.3 URL vs tipo de ataque - Top 15 URLs

top_urls = datos["url"].value_counts().head(15).index
tabla_url_ataque = pd.crosstab(
    datos[datos["url"].isin(top_urls)]["url"],
    datos[datos["url"].isin(top_urls)]["tipo_ataque"]
)
grafico_heatmap(tabla_url_ataque, "Top URLs vs tipo de ataque", "21_heatmap_url_vs_tipo_ataque.png")

In [ ]:
# 8.4 IP vs tipo de ataque - Top 15 IPs atacantes

top_ips_ataque = ataques["ip_origen"].value_counts().head(15).index
tabla_ip_ataque = pd.crosstab(
    ataques[ataques["ip_origen"].isin(top_ips_ataque)]["ip_origen"],
    ataques[ataques["ip_origen"].isin(top_ips_ataque)]["tipo_ataque"]
)

if not tabla_ip_ataque.empty:
    grafico_heatmap(tabla_ip_ataque, "Top IPs atacantes vs tipo de ataque", "22_heatmap_ip_vs_tipo_ataque.png")
else:
    print("No hay ataques para graficar IP vs tipo de ataque.")

In [ ]:
# 8.5 Barras apiladas: tipo de ataque por método HTTP

tabla_metodo_ataque.plot(kind="bar", stacked=True, figsize=(12, 6))
plt.title("Tipo de ataque por método HTTP")
plt.xlabel("Método HTTP")
plt.ylabel("Cantidad de eventos")
plt.xticks(rotation=0)
guardar_mostrar("23_barras_apiladas_tipo_ataque_por_metodo.png")

In [ ]:
# 8.6 Barras apiladas: tipo de ataque por código de estado

tabla_estado_ataque.plot(kind="bar", stacked=True, figsize=(12, 6))
plt.title("Tipo de ataque por código de estado HTTP")
plt.xlabel("Código de estado")
plt.ylabel("Cantidad de eventos")
plt.xticks(rotation=0)
guardar_mostrar("24_barras_apiladas_tipo_ataque_por_estado.png")

## 9. Indicadores visuales adicionales

In [ ]:
# 9.1 Porcentaje de ataques vs normales - gráfico circular

conteo_clase = datos["es_ataque_label"].value_counts()

plt.figure(figsize=(8, 8))
plt.pie(conteo_clase.values, labels=conteo_clase.index, autopct="%1.1f%%", startangle=90)
plt.title("Porcentaje de eventos normales vs ataques")
guardar_mostrar("25_pie_porcentaje_normales_vs_ataques.png")

In [ ]:
# 9.2 Porcentaje por tipo de ataque - gráfico circular

conteo_tipo = datos["tipo_ataque"].value_counts()

plt.figure(figsize=(9, 9))
plt.pie(conteo_tipo.values, labels=conteo_tipo.index, autopct="%1.1f%%", startangle=90)
plt.title("Porcentaje por tipo de ataque")
guardar_mostrar("26_pie_porcentaje_tipo_ataque.png")

In [ ]:
# 9.3 Serie temporal por minuto

eventos_minuto = datos.groupby("minuto").size()

plt.figure(figsize=(14, 6))
eventos_minuto.plot(kind="line")
plt.title("Eventos por minuto")
plt.xlabel("Fecha y hora")
plt.ylabel("Cantidad de eventos")
plt.xticks(rotation=45)
guardar_mostrar("27_eventos_por_minuto.png")

In [ ]:
# 9.4 Ataques por minuto

ataques_minuto = ataques.groupby("minuto").size()

plt.figure(figsize=(14, 6))
ataques_minuto.plot(kind="line")
plt.title("Ataques por minuto")
plt.xlabel("Fecha y hora")
plt.ylabel("Cantidad de ataques")
plt.xticks(rotation=45)
guardar_mostrar("28_ataques_por_minuto.png")

In [ ]:
# 9.5 Tamaño promedio de respuesta por tipo de ataque

promedio_tamano = datos.groupby("tipo_ataque")["tamano_respuesta"].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
promedio_tamano.plot(kind="bar")
plt.title("Tamaño promedio de respuesta por tipo de ataque")
plt.xlabel("Tipo de ataque")
plt.ylabel("Tamaño promedio de respuesta en bytes")
plt.xticks(range(len(promedio_tamano.index)), envolver_etiquetas(promedio_tamano.index), rotation=45, ha="right")
guardar_mostrar("29_tamano_promedio_respuesta_por_tipo_ataque.png")

In [ ]:
# 9.6 Boxplot de tamaño de respuesta por tipo de ataque

tipos = datos["tipo_ataque"].dropna().unique()
series = [datos.loc[datos["tipo_ataque"] == t, "tamano_respuesta"].dropna() for t in tipos]

plt.figure(figsize=(14, 7))
plt.boxplot(series, tick_labels=envolver_etiquetas(tipos, 14))
plt.title("Boxplot de tamaño de respuesta por tipo de ataque")
plt.xlabel("Tipo de ataque")
plt.ylabel("Tamaño de respuesta en bytes")
plt.xticks(rotation=45, ha="right")
guardar_mostrar("30_boxplot_tamano_respuesta_por_tipo_ataque.png")

## 10. Resumen automático de gráficos generados

In [ ]:
# ============================================================
# 10. Listado final de archivos PNG generados
# ============================================================

graficos_generados = sorted(DIR_GRAFICOS.glob("*.png"))

print(f"Total de gráficos generados: {len(graficos_generados)}")
for grafico in graficos_generados:
    print(grafico)

## 11. Interpretación sugerida

Al revisar los gráficos, presta especial atención a:

1. **IPs con mayor número de ataques:** posibles fuentes maliciosas o direcciones para enriquecer con inteligencia de amenazas.
2. **URLs más atacadas:** rutas críticas como `/login`, `/admin`, `/dashboard` o endpoints de API.
3. **Horas con mayor actividad maliciosa:** útil para reforzar monitoreo y reglas de correlación.
4. **Códigos HTTP 401, 403, 404 y 500:** pueden indicar fuerza bruta, escaneo, errores explotables o reconocimiento.
5. **User-Agent anómalos:** herramientas automatizadas, bots o scripts.
6. **Tipos de ataque dominantes:** base para priorizar controles, reglas SIEM, WAF o hardening.